# 🦴 UniRig — Auto-rig NEURAL (esqueleto + skinning con IA) — gratis en Colab

Reemplaza el auto-rig heurístico (que **deforma feo los brazos**) por el rig neural de **UniRig** (VAST-AI, los mismos de Hunyuan3D). Entrás con un `.glb`/`.obj`/`.fbx` (el arquero de Hunyuan) y salís con un **`.glb` riggeado** (esqueleto + pesos de calidad pro).

## Antes de empezar
- **GPU:** `Entorno de ejecución` -> `Cambiar tipo de entorno` -> **T4 GPU**.
- UniRig **exige Python 3.11** (por `bpy`/Blender). Colab hoy trae Python 3.12, asi que la **Celda 0** baja a 3.11 con `condacolab` (reinicia el kernel solo: el mensaje 'Your session crashed' es NORMAL).
- **FlashAttention no corre en la T4** (necesita GPU Ampere+); no se instala. UniRig usa atencion estandar.

## Orden de las celdas
1. **Celda 0** -> baja a Python 3.11 (reinicia solo). Corre ESTA PRIMERO.
2. **Celda 1** -> instala UniRig (~8-12 min).
3. **Celda 2** -> subis tu modelo 3D.
4. **Celda 3** -> genera el rig (esqueleto -> skin -> merge).
5. **Celda 4** -> descargas el `.glb` riggeado.
6. **Celda 5** (opcional) -> imprime los nombres de los huesos (pasamelos para el retarget del mocap CMU).


## Celda 0 - Python 3.11 (obligatorio). Reinicia el kernel solo.


In [ ]:
# Celda 0 - corre ESTA PRIMERO. El kernel se reinicia solo ('Your session crashed' = NORMAL).
!pip install -q condacolab
import condacolab
condacolab.install()   # instala Miniforge con Python 3.11 y reinicia el entorno

## Celda 1 - Instalar UniRig (stack fijo probado)
Tarda ~8-12 min (la primera vez baja torch, spconv, Blender). **No** hace falta reiniciar al terminar.


In [ ]:
# Celda 1 - Instalar UniRig. Corre DESPUES de que la Celda 0 reinicie el kernel.
import sys, os
print('Python:', sys.version.split()[0], '(debe ser 3.11.x)')
assert sys.version_info[:2] == (3, 11), 'No estas en Python 3.11 -> corre la Celda 0 y espera el reinicio.'
!nvidia-smi -L

os.chdir('/content')
if not os.path.isdir('/content/UniRig'):
    !git clone https://github.com/VAST-AI-Research/UniRig.git
os.chdir('/content/UniRig')

# 1) PyTorch 2.3.1 + CUDA 12.1 (compatible con la T4)
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121

# 2) numpy que pide UniRig (los warnings de otros paquetes sobre numpy>=2 son inofensivos)
!pip install -q numpy==1.26.4

# 3) spconv para CUDA 12.x (wheel cp311)
!pip install -q spconv-cu120

# 4) torch_scatter / torch_cluster (wheels PyG para torch 2.3.1 + cu121)
!pip install -q torch_scatter torch_cluster -f https://data.pyg.org/whl/torch-2.3.1+cu121.html

# 5) Blender headless (bpy 4.2 solo existe para Python 3.11)
!pip install -q bpy==4.2

# 6) Resto de dependencias de UniRig
!pip install -q -r requirements.txt 2>&1 | tail -3

# 7) Re-fijar numpy por si algun requirement lo movio
!pip install -q numpy==1.26.4

print('\nVerificando imports clave...')
import torch, numpy
import spconv, torch_scatter, torch_cluster, bpy  # noqa
print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
print('numpy', numpy.__version__, '| bpy', bpy.app.version_string)
print('\nUniRig instalado. Pasa a la Celda 2.')

## Celda 2 - Subi tu modelo 3D
Subi el `.glb` del arquero (el de Hunyuan) o cualquier `.obj`/`.fbx`/`.glb`/`.vrm`.


In [ ]:
# Celda 2 - Subir modelo (define MODEL)
import os
os.chdir('/content/UniRig')
from google.colab import files
up = files.upload()
name = list(up.keys())[0]
MODEL = os.path.abspath(name)
print('Modelo:', MODEL, '|', round(os.path.getsize(MODEL)/1024, 1), 'KB')

## Celda 3 - Auto-rig neural (esqueleto -> skin -> merge)
Tres pasos de UniRig. Tarda unos minutos (la primera vez baja los pesos del modelo desde HuggingFace).


In [ ]:
# Celda 3 - Generar el rig
import os
os.chdir('/content/UniRig')
NAME  = os.path.splitext(os.path.basename(MODEL))[0]
SKEL  = f'/content/results/{NAME}_skeleton.fbx'
SKIN  = f'/content/results/{NAME}_skin.fbx'
RIGGED = f'/content/{NAME}_rigged.glb'
os.makedirs('/content/results', exist_ok=True)

print('== 1/3 Esqueleto =='); get_ipython().system('bash launch/inference/generate_skeleton.sh --input "{}" --output "{}"'.format(MODEL, SKEL))
print('== 2/3 Skinning ==');  get_ipython().system('bash launch/inference/generate_skin.sh --input "{}" --output "{}"'.format(SKEL, SKIN))
print('== 3/3 Merge ==');     get_ipython().system('bash launch/inference/merge.sh --source "{}" --target "{}" --output "{}"'.format(SKIN, MODEL, RIGGED))

print('\nRiggeado:', RIGGED, '| existe:', os.path.exists(RIGGED))

## Celda 4 - Descargar el `.glb` riggeado


In [ ]:
# Celda 4 - Descargar
from google.colab import files
files.download(RIGGED)

## Celda 5 (opcional) - Nombres de los huesos
Imprime la lista de huesos que genero UniRig. **Copiamelos y te armo el mapa de retarget** para animar el arquero con el mocap CMU (caminar/correr/saltar).


In [ ]:
# Celda 5 (opcional) - Listar huesos del rig
!pip install -q pygltflib
from pygltflib import GLTF2
g = GLTF2().load(RIGGED)
huesos = [n.name for n in g.nodes if n.name]
print(f'{len(huesos)} nodos/huesos:')
print('\n'.join(huesos))

---
### Si algo falla
- **`No matching distribution ... bpy==4.2` o `spconv-cu120`** -> seguis en Python 3.12. Volve a la **Celda 0** y espera el reinicio antes de la Celda 1.
- **Error de `flash_attn` / `sm_75` / atencion** -> la T4 no soporta FlashAttention. Necesitarias una GPU Ampere+ (Colab Pro con A100) o correr UniRig por API (Replicate/fal, centavos por modelo).
- **`Model path not found` / descarga de pesos** -> reintenta la Celda 3 (a veces la descarga de HuggingFace se corta).
- Cualquier error rojo: copialo y pasamelo.
